Using StanfordNLP tool to transform the data into Sentiment Annotated Tree format https://stanfordnlp.github.io/CoreNLP/sentiment.html.


In [1]:
#imports
import pandas as pd
import logging
import json
import re
from stanfordcorenlp import StanfordCoreNLP
from nltk.tokenize import sent_tokenize

In [2]:
#transform StanforNLPCore java code into python code
# java -mx4g -cp "*" edu.stanford.nlp.pipeline.StanfordCoreNLPServer -port 9000 -timeout 1500000
class StanfordNLP:
    def __init__(self, host='http://localhost', port=9000):
        self.nlp = StanfordCoreNLP(host, port=port,
                                   timeout=30000)  # , quiet=False, logging_level=logging.DEBUG)
        self.props = {
            'annotators': 'sentiment',
            'pipelineLanguage': 'en',
            'outputFormat': 'json'
        }

    def word_tokenize(self, sentence):
        return self.nlp.word_tokenize(sentence)

    def pos(self, sentence):
        return self.nlp.pos_tag(sentence)

    def ner(self, sentence):
        return self.nlp.ner(sentence)

    def parse(self, sentence):
        return self.nlp.parse(sentence)

    def dependency_parse(self, sentence):
        return self.nlp.dependency_parse(sentence)

    def annotate(self, sentence):
        return json.loads(self.nlp.annotate(sentence, properties=self.props))
    
    def sent_anot(self, sentence):
        return dir(self.nlp)

    @staticmethod
    def tokens_to_dict(_tokens):
        tokens = defaultdict(dict)
        for token in _tokens:
            tokens[int(token['index'])] = {
                'word': token['word'],
                'lemma': token['lemma'],
                'pos': token['pos'],
                'ner': token['ner']}
        return tokens

In [3]:
#load data
data_train = pd.read_csv("data\train.csv")
data_test = pd.read_csv("data\test.csv")

In [4]:
data_train.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [5]:
#get comments text from data
X_train = data_train.iloc[:,1].values
X_test = data_test.iloc[:,1].values

In [6]:
def clean_complete(text):
    """
    removes all the symbols which are not alphabetical or punctuation from text,
    replaces paragraph with dot
    """
    text = text.lower() #lower case
    for char in text:
        if char == "\n":
            text = text.replace(char, ".")
        elif char not in "abcdefghijklmnopqrstuvwxyz.!? ":
            text = text.replace(char, " ")
    return text

In [7]:
def transform_to_SentAnTree(sentence):
    """
    transforms a sentence to SAT
    """
    #convert a sentence to a tree
    annotation = sNLP.annotate(sentence)
    sent_tree = annotation['sentences'][0]['sentimentTree']
    #remove unnecessary information
    result = re.sub(r'[@\w\.$\-:`]+\|sentiment=(\d)\|prob=[01].\d+', r'\1', sent_tree)
    return re.sub("\s+"," ", result) #reduce additional spaces

In [8]:
sNLP = StanfordNLP()

small_df = pd.DataFrame(columns =["comment_number", "comment_sentence"] )
index_count = 0
comment_count = 0

In [ ]:
#transform train data
for comment in X_train:
    
    #clean the comment
    comment_clean = clean_complete(comment)
    #divide to sentences
    sentences = sent_tokenize(comment_clean)
    for s in sentences:
        #for very long sentences
        if len(s) > 1000:
            s = s[:1000]
        try:
            tree_sentence = transform_to_SentAnTree(s)
            
            small_df.loc[index_count] = [comment_count, tree_sentence]
            index_count += 1
        except :
            print(comment_count, "some issue")
            continue
            
    comment_count += 1
    if comment_count % 100 == 0:
        #save the results
        results_csv = small_df.to_csv("translated_data_test.csv", index=True)
        print("Saved", comment_count)

2 some issue


In [ ]:
#transform test data
for comment in X_test:
    
    #clean the comment
    comment_clean = clean_complete(comment)
    #divide to sentences
    sentences = sent_tokenize(comment_clean)
    for s in sentences:
        #for very long sentences
        if len(s) > 1000:
            s = s[:1000]
        try:
            tree_sentence = transform_to_SentAnTree(s)
            
            small_df.loc[index_count] = [comment_count, tree_sentence]
            index_count += 1
        except :
            print(comment_count, "some issue")
            continue
            
    comment_count += 1
    if comment_count % 100 == 0:
        #save the results
        results_csv = small_df.to_csv("translated_data_test.csv", index=True)
        print("Saved", comment_count)